# Portfolio Optimizer — Demo

Demonstrates the full pipeline: data loading, optimization (GMV / Max-Sharpe / efficient frontier), risk evaluation, stress testing, forward stepwise asset selection, and comparing portfolios.

See `../README.md` for methodology notes, metric interpretation guidance, and known limitations.

In [10]:
import sys
sys.path.append("..")

from portfolio_optimizer import (
    run_portfolio_analysis,
    forward_stepwise_selection,
    compare_all_portfolios,
)

## 1. Build a manually chosen portfolio

Loads data, solves for the GMV and Max-Sharpe portfolios and runs the full risk evaluation + stress test. `plot_frontier` if True, plots the efficient frontier. `save_as` registers this portfolio for the comparison in section 4.

In [11]:
results = run_portfolio_analysis(["VXUS", "BND", "GLD", "VNQ"], save_as="auto",plot_frontier=False)

[*********************100%***********************]  4 of 4 completed

Asset-level statistics (annualized):
                   BND     GLD     VNQ    VXUS
Expected Return  2.31%   8.66%   9.29%   8.18%
Volatility       4.86%  16.77%  19.81%  17.71%

Portfolio weights:
                      BND     GLD     VNQ   VXUS
GMV Weight         94.37%   0.00%   0.00%  5.63%
Max-Sharpe Weight   0.00%  52.77%  40.28%  6.95%

Portfolio summary:
           Expected Return Volatility Sharpe
GMV                  2.64%      4.76%  -0.43
Max-Sharpe           8.88%     13.27%   0.32

RISK EVALUATION — Max-Sharpe portfolio

Expected return: 8.88% | Volatility: 13.27% | Sharpe: 0.32

--- Tail risk (95% confidence, $1,000,000 portfolio) ---
Parametric VaR:   $129,538  (12.95%)
Parametric CVaR:  $185,002  (18.50%)
Historical VaR:   $201,141  (20.11%)
Historical CVaR:  $317,014  (31.70%)

--- Path risk ---
Max drawdown: -23.71%

--- Risk-adjusted return ratios ---
Sortino ratio: 0.40   Calmar ratio: 0.37

--- Distribution shape ---
Skewness: -0.57   Excess kurtosis: 8.87

--- Di

## 2. Select assets from a larger candidate universe

Greedy forward stepwise selection: starts with the single best asset, then repeatedly adds whichever remaining candidate improves the Max-Sharpe ratio the most, until no candidate improves it by at least `min_improvement` (or `max_assets` is reached).

`min_improvement` matters here: without a real threshold, the algorithm would keep adding assets even for negligible or purely numerical-noise gains, since adding an asset can never mathematically *hurt* the achievable Sharpe ratio in a long-only optimizer — only a minimum-improvement cutoff actually stops it at a meaningful point.

In [12]:
candidates = ["VXUS", "BND", "GLD", "VNQ", "TLT", "DBC", "VWO", "SHY"]
selected_tickers, selection_history = forward_stepwise_selection(candidates)

[*********************100%***********************]  8 of 8 completed


FORWARD STEPWISE SELECTION
(minimum Sharpe improvement to keep adding: 0.005)
Step 1: added GLD      -> Sharpe = 0.237 (+inf)
Step 2: added VNQ      -> Sharpe = 0.315 (+0.0781)

Best remaining candidate (VXUS) only improves Sharpe by 0.0007, below the 0.005 threshold — stopping.

Final selected assets: ['GLD', 'VNQ']
Final Sharpe ratio: 0.315
(See README.md "Interpreting the risk metrics" for guidance on reading the per-step improvement.)


## 3. Evaluate the selected portfolio

Feed the output of forward selection straight back into the pipeline — again saved for the comparison below.

In [13]:
selected_results = run_portfolio_analysis(selected_tickers, save_as="auto")

[*********************100%***********************]  2 of 2 completed

Asset-level statistics (annualized):
                    GLD     VNQ
Expected Return   9.22%  10.55%
Volatility       16.76%  20.29%

Portfolio weights:
                      GLD     VNQ
GMV Weight         60.63%  39.37%
Max-Sharpe Weight  52.72%  47.28%

Portfolio summary:
           Expected Return Volatility Sharpe
GMV                  9.74%     13.61%   0.37
Max-Sharpe           9.85%     13.75%   0.38

RISK EVALUATION — Max-Sharpe portfolio

Expected return: 9.85% | Volatility: 13.75% | Sharpe: 0.38

--- Tail risk (95% confidence, $1,000,000 portfolio) ---
Parametric VaR:   $127,685  (12.77%)
Parametric CVaR:  $185,147  (18.51%)
Historical VaR:   $213,512  (21.35%)
Historical CVaR:  $328,112  (32.81%)

--- Path risk ---
Max drawdown: -24.39%

--- Risk-adjusted return ratios ---
Sortino ratio: 0.48   Calmar ratio: 0.40

--- Distribution shape ---
Skewness: -0.55   Excess kurtosis: 8.07

--- Diversification ---
Diversification ratio: 1.34

(See README.md "Interpreting the risk metri

## 4. Compare all saved portfolios

Pulls together every portfolio saved via `save_as` above into one side-by-side table. Add more `run_portfolio_analysis(..., save_as="...")` calls above (e.g. a different manual mix) to include them here too — no separate comparison call needed per pair.

In [14]:
compare_all_portfolios()

Comparison (Max-Sharpe portfolios):
                      BND+GLD+VNQ+VXUS   GLD+VNQ
Expected Return                  8.88%     9.85%
Volatility                      13.27%    13.75%
Sharpe                            0.32      0.38
Parametric VaR                $129,538  $127,685
Parametric CVaR               $185,002  $185,147
Historical VaR                $201,141  $213,512
Historical CVaR               $317,014  $328,112
Max Drawdown                   -23.71%   -24.39%
Sortino Ratio                     0.40      0.48
Calmar Ratio                      0.37      0.40
Skewness                         -0.57     -0.55
Excess Kurtosis                   8.87      8.07
Diversification Ratio             1.36      1.34
# Assets                             4         2

Highest Sharpe: GLD+VNQ (0.375)
